# Second Look: SmolVLA fine-tune on Kaggle (LeRobot 0.6.1, native v3.0 dataset)

**Before you run anything** (right-hand sidebar):
- **Session options > Accelerator:** `GPU T4 x2` (only one is used). `P100` also works; the install cell picks the right torch build.
- **Session options > Internet:** `On` (pip, the SmolVLA base model download and uv's Python download all need it).
- **Input > Add Input:** your dataset `second-look-v2` (the zip you uploaded; Kaggle unzips it).
- **Add-ons > Secrets:** `HF_TOKEN` (optional, only to push the model to the Hub).

Run cells top to bottom. Each section ends with a clear PASS line.

**Folder layout this notebook creates in `/kaggle/working`:**
```
lr_env/                 isolated Python 3.12 + LeRobot (deleted at the end)
second_look_v2/         writable copy of the dataset (deleted at the end)
runs/smoke/             30-step smoke test (deleted at the end)
runs/smolvla_run1/      full training run + checkpoints
final_model/            the ONLY thing kept as output: last checkpoint
```
The dataset is already v3.0, so there is no conversion step.

In [ ]:
import subprocess, os, sys, glob, json, shutil, re

WORK    = "/kaggle/working"
ENV     = f"{WORK}/lr_env"
PY      = f"{ENV}/bin/python"
DS_ROOT = f"{WORK}/second_look_v2"          # writable copy of the dataset
REPO_ID = "local/second_look_v2"            # just a label; nothing is uploaded
OUT_DIR = f"{WORK}/runs/smolvla_run1"
FINAL   = f"{WORK}/final_model"

def sh(cmd, check=True, show=True):
    """Run a shell command, stream output into the notebook, return the full output text."""
    if show: print("$", cmd[:300], flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    buf = []
    for line in p.stdout:
        buf.append(line)
        if show: print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd[:200]}")
    return "".join(buf)

print("system python:", sys.version.split()[0])
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
sh(f"df -h {WORK} | tail -n 1")

## 1. Find and check the dataset

In [ ]:
cands = []
for h in glob.glob("/kaggle/input/**/meta/info.json", recursive=True):
    try:
        i = json.load(open(h))
    except Exception:
        continue
    cands.append((h, i))
assert cands, "No meta/info.json under /kaggle/input. Add the dataset via Input > Add Input."
v3 = [(h, i) for h, i in cands if i.get("codebase_version") == "v3.0"]
assert v3, f"Found datasets but none are v3.0: {[(h, i.get('codebase_version')) for h, i in cands]}"
if len(v3) > 1:
    print("WARNING: several v3.0 datasets found, using the first:", [h for h, _ in v3])
hit, info = v3[0]
SRC = os.path.dirname(os.path.dirname(hit))
print("dataset root     :", SRC)
print("codebase_version :", info["codebase_version"])
print("episodes / frames:", info["total_episodes"], "/", info["total_frames"], " fps:", info["fps"])
for k, v in info["features"].items():
    print(f"  {k:30s} {v['dtype']:8s} {v['shape']}")
assert info["total_episodes"] == 1200, f"expected 1200 episodes, got {info['total_episodes']}"
assert info["features"]["observation.state"]["shape"] == [6] and info["features"]["action"]["shape"] == [6]
for cam in ("observation.images.front", "observation.images.wrist"):
    assert cam in info["features"], f"missing {cam}"
sh(f"du -sh {SRC}")

# Work on a writable copy so nothing in /kaggle/input is ever touched (it is only ~170 MB).
shutil.rmtree(DS_ROOT, ignore_errors=True)
shutil.copytree(SRC, DS_ROOT)
print("DATASET FOUND + COPIED ->", DS_ROOT)

## 2. Build the Python 3.12 environment (about 5-10 minutes)

Same approach as before: Kaggle's own Python/torch conflicts with LeRobot 0.6, so we build an isolated venv with a torch build that matches Kaggle's GPU driver.

In [ ]:
drv = sh("nvidia-smi --query-gpu=driver_version --format=csv,noheader", show=False).strip().splitlines()[0]
drv_major = int(drv.split(".")[0])
assert drv_major >= 525, f"Driver {drv} is too old for any current PyTorch CUDA build."
cc = float(sh("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", show=False).strip().splitlines()[0])
CU = "cu128" if (drv_major >= 570 and cc >= 7.0) else "cu126"
print(f"driver {drv}, compute capability {cc} -> using PyTorch index {CU}")

sh("pip install -q uv")
sh(f"rm -rf {ENV}")
sh(f"uv venv --python 3.12 {ENV}")
# 1) torch first, from the index matching the driver (stops pip pulling a CUDA 13 build)
sh(f"uv pip install --python {PY} 'torch>=2.7,<2.12' 'torchvision>=0.22,<0.27' --index-url https://download.pytorch.org/whl/{CU}")
# 2) LeRobot + SmolVLA + training extras (keeps the torch already installed)
sh(f"uv pip install --python {PY} 'lerobot[smolvla,training]==0.6.1'")

### 2b. Verify the environment (must print `ENV OK`)

In [ ]:
VERIFY = r"""
import sys, importlib.metadata as md, torch
print("python", sys.version.split()[0])
print("torch", torch.__version__, "| CUDA build", torch.version.cuda, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "torch cannot see the GPU. Usually a CUDA-13 torch got installed. Fix: re-run section 2 "
    "(or `uv pip install --python lr_env/bin/python --reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu126`).")
x = torch.randn(2048, 2048, device="cuda"); print("matmul on", torch.cuda.get_device_name(0), float((x @ x).sum()) != 0)
for p in ["lerobot", "transformers", "datasets", "accelerate", "av", "num2words", "numpy"]:
    print(f"{p:14s}", md.version(p))
import lerobot.policies.smolvla.modeling_smolvla
print("ENV OK")
"""
open(f"{WORK}/verify_env.py", "w").write(VERIFY)
sh(f"{PY} {WORK}/verify_env.py")

## 3. Smoke-load the dataset through LeRobot (must print `DATASET OK`)

In [ ]:
sh(f"""{PY} - <<'EOF'
from lerobot.datasets import LeRobotDataset
ds = LeRobotDataset("{REPO_ID}", root="{DS_ROOT}", video_backend="pyav")
print("episodes", ds.num_episodes, "| frames", ds.num_frames, "| fps", ds.fps)
assert ds.num_episodes == 1200
for i in (0, len(ds) // 2, len(ds) - 1):
    item = ds[i]
    print(i, {{k: tuple(v.shape) for k, v in item.items() if hasattr(v, "shape")}})
    print("   task:", item.get("task"))
print("DATASET OK")
EOF""")

## 4. Smoke train (about 3-5 minutes), then read the speed

Camera mapping: SmolVLA-base expects `camera1/camera2/camera3`; our dataset has `front` and `wrist`, so we rename them and add one masked empty camera.

In [ ]:
RENAME = {"observation.images.front": "observation.images.camera1",
          "observation.images.wrist": "observation.images.camera2"}

def train_cmd(steps, batch_size, out_dir, save_freq, job="second_look_smolvla"):
    return (f"{ENV}/bin/lerobot-train"
            f" --dataset.repo_id={REPO_ID} --dataset.root={DS_ROOT} --dataset.video_backend=pyav"
            f" --policy.path=lerobot/smolvla_base --policy.device=cuda --policy.push_to_hub=false"
            f" --policy.empty_cameras=1 --rename_map='{json.dumps(RENAME)}'"
            f" --batch_size={batch_size} --steps={steps} --save_freq={save_freq}"
            f" --num_workers=2 --log_freq=10 --seed=1000"
            f" --output_dir={out_dir} --job_name={job} --wandb.enable=false")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
BATCH = 16     # T4 16 GB: 16 is safe; try 32 after the smoke run if memory allows
sh(f"rm -rf {WORK}/runs/smoke")
out = sh(train_cmd(steps=30, batch_size=BATCH, out_dir=f"{WORK}/runs/smoke", save_freq=30, job="smoke"))

upd = [float(x) for x in re.findall(r"updt_s:([\d.]+)", out)]
dat = [float(x) for x in re.findall(r"data_s:([\d.]+)", out)]
if upd and dat:
    sec_per_step = max(upd) + max(dat)       # conservative
    print(f"\n~{sec_per_step:.2f} s/step at batch {BATCH}  ->  {3600/sec_per_step:.0f} steps/hour")
    for hours in (4, 8, 10):
        print(f"  {hours:>2} h budget ~ {int(hours*3600/sec_per_step):,} steps")
else:
    print("Could not parse speed from the log; read the `updt_s` / `data_s` values above by eye.")

## 5. Full training

Set `STEPS` from the budget table above. Keep headroom: a session is capped at 12 h and the weekly GPU quota is 30 h. For scale: 1 epoch of this dataset is 99,240 frames, i.e. about 6,200 steps at batch 16.

**To run unattended:** *Save Version > Save & Run All (Commit)*. It keeps running with the browser closed. The last cell of this notebook cleans up the big folders so only `final_model/` is saved as output.

In [ ]:
STEPS     = 10000        # <- edit using the budget table from section 4
SAVE_FREQ = max(STEPS // 4, 500)
sh(f"rm -rf {OUT_DIR}")
sh(train_cmd(steps=STEPS, batch_size=BATCH, out_dir=OUT_DIR, save_freq=SAVE_FREQ))

### If the session dies: resume from the last checkpoint
Only works if `/kaggle/working` still has the run (a crash inside the same session). Uncomment and run.

In [ ]:
# sh(f"{ENV}/bin/lerobot-train --resume=true --config_path={OUT_DIR}/checkpoints/last/pretrained_model/train_config.json")

## 6. Save the result and clean up

Copies the last checkpoint to `final_model/`, then (if `CLEANUP = True`) deletes the venv, dataset copy, smoke run and intermediate checkpoints. Without this, a committed run would save about 8-10 GB of environment as notebook output.

In [ ]:
last = f"{OUT_DIR}/checkpoints/last/pretrained_model"
assert os.path.isdir(last), f"No final checkpoint at {last}. Did training finish?"
shutil.rmtree(FINAL, ignore_errors=True)
shutil.copytree(last, FINAL)
print(sh(f"ls -la {FINAL}"))
assert os.path.exists(f"{FINAL}/model.safetensors"), "model.safetensors missing from final_model/"
print("FINAL MODEL SAVED ->", FINAL)

# Optional: push to your Hugging Face account so it survives the Kaggle session.
HF_MODEL_REPO = None    # e.g. "your-hf-username/second-look-smolvla"
if HF_MODEL_REPO:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    sh(f"""{PY} - <<'EOF'
from huggingface_hub import HfApi
api = HfApi()
api.create_repo("{HF_MODEL_REPO}", exist_ok=True, private=True)
api.upload_folder(repo_id="{HF_MODEL_REPO}", folder_path="{FINAL}")
print("uploaded")
EOF""")

CLEANUP = True          # only runs if the final model was saved above
if CLEANUP:
    for p in (ENV, DS_ROOT, f"{WORK}/runs"):
        shutil.rmtree(p, ignore_errors=True)
    for f in ("verify_env.py",):
        try: os.remove(f"{WORK}/{f}")
        except FileNotFoundError: pass
    sh(f"du -sh {WORK}/* ; df -h {WORK} | tail -n 1")